# 02 – Data Cleaning
Goal: apply the decisions from `01_data_review` and save clean tables to `data/processed` for loading into PostgreSQL.

Decisions applied:
1. Convert dates to datetime and correct the 2000-day shift in `Ship Date`
2. Keep US orders only (Canadian orders have no coordinates)
3. Rename columns to snake_case (SQL-friendly)
4. Keep only the needed columns from the ZIP table

In [1]:
import pandas as pd

RAW = "../data/raw/"
PROCESSED = "../data/processed/"

sales     = pd.read_csv(RAW + "Candy_Sales.csv", dtype={"Postal Code": str})
products  = pd.read_csv(RAW + "Candy_Products.csv")
factories = pd.read_csv(RAW + "Candy_Factories.csv")
zips      = pd.read_csv(RAW + "uszips.csv", dtype={"zip": str})

print("Raw sales rows:", len(sales))

Raw sales rows: 10194


## 1. Dates
Convert to datetime and subtract the 2000-day shift from `Ship Date`.

In [2]:
sales["Order Date"] = pd.to_datetime(sales["Order Date"])
sales["Ship Date"] = pd.to_datetime(sales["Ship Date"]) - pd.Timedelta(days=2000)
sales["Lead Time Days"] = (sales["Ship Date"] - sales["Order Date"]).dt.days

assert sales["Lead Time Days"].min() >= 0, "Ship date before order date!"

print("Ship Date:", sales["Ship Date"].min().date(), "→", sales["Ship Date"].max().date())
print(sales["Lead Time Days"].describe())

Ship Date: 2021-01-07 → 2025-01-05
count    10194.000000
mean         3.961840
std          1.744106
min          0.000000
25%          3.000000
50%          4.000000
75%          5.000000
max         11.000000
Name: Lead Time Days, dtype: float64


## 2. US orders only
Canadian orders have no coordinates in the ZIP table → excluded (Project Charter scope).

In [3]:
rows_before = len(sales)
sales = sales[sales["Country/Region"] == "United States"].copy()

assert len(sales) == 9994, "Unexpected number of US orders"
print(f"Removed {rows_before - len(sales)} Canadian rows, {len(sales)} US rows left")

Removed 200 Canadian rows, 9994 US rows left


## 3. Column names
Rename columns to snake_case so they can be used in SQL without quotes.

In [4]:
def to_snake_case(df):
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df

for df in [sales, products, factories]:
    to_snake_case(df)

print("sales:", sales.columns.tolist())
print("products:", products.columns.tolist())
print("factories:", factories.columns.tolist())

sales: ['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'country_region', 'city', 'state_province', 'postal_code', 'division', 'region', 'product_id', 'product_name', 'sales', 'units', 'gross_profit', 'cost', 'lead_time_days']
products: ['division', 'product_name', 'factory', 'product_id', 'unit_price', 'unit_cost']
factories: ['factory', 'latitude', 'longitude']


## 4. ZIP table
Keep only the columns needed for distance calculation and only the ZIP codes that appear in sales.

In [5]:
zips = zips[["zip", "lat", "lng", "city", "state_id"]]
zips = zips[zips["zip"].isin(sales["postal_code"])].copy()

assert sales["postal_code"].isin(zips["zip"]).all(), "Some sales ZIP codes have no coordinates"
print(f"ZIP table reduced to {len(zips)} rows (codes used in sales)")

ZIP table reduced to 631 rows (codes used in sales)


## 5. Save clean tables

In [6]:
import os
os.makedirs(PROCESSED, exist_ok=True)

sales.to_csv(PROCESSED + "sales.csv", index=False)
products.to_csv(PROCESSED + "products.csv", index=False)
factories.to_csv(PROCESSED + "factories.csv", index=False)
zips.to_csv(PROCESSED + "zips.csv", index=False)

print(os.listdir(PROCESSED))

['factories.csv', 'products.csv', 'sales.csv', 'zips.csv']
